# 16.8 手寫 attention 怎麼換成最佳化介面？

# 第 16 章：由量測帶出效率 BKM

前置：已有第4章能forward與生成的模型。像整理廚房，先找哪一步最慢再改工作流程。CPU小例只驗證數值與資料契約；GPU速度、峯值記憶體與編譯收益必須在讀者的硬體實測。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：不能偷看明天的答案**

對角線可看；右上角未來位置禁止。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/causal.svg")))

**先想一想：**手寫mask裏的True是允許，傳SDPA時要反轉嗎？

先手算做對，再換同數學條件的SDPA。它可能選不同後端；True在這個API代表「允許看」，不一定和其他mask API一致。

**把直覺寫成數學：**SDPA默認內部縮放QKᵀ/√D；不要提前縮一次又讓它再縮一次。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.attention import manual_attention

q, k, v = [torch.randn(1, 1, 4, 3) for _ in range(3)]
allowed = torch.ones(4, 4, dtype=torch.bool).tril()[None, None]
manual, _ = manual_attention(q, k, v, allowed)
optimized = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed, dropout_p=0.0)
print((manual - optimized).abs().max().item())
assert torch.allclose(manual, optimized, atol=1e-6)

**如何讀結果：**dropout_p=0在兩路固定；模型梯度等價另由核心tests驗證。

**只改一件事：**只把mask反轉，說明爲什麼結果不再相同。
